*Notebook trước: [5. Data Visualization](./5.%20Data%20Visualization.ipynb)*

# Bước 6: Model Building & Evaluation — Tối Giản 2 Bảng Chuẩn & 3 Kịch Bản
### Đề án: Phân Tích Hành Vi Mua Hàng Trong Thương Mại Điện Tử (ADY201m)
---
> **Thiết kế mô hình tinh gọn & chống rò rỉ dữ liệu (No Data Leakage):**
> - **Đặc trưng đầu vào (`X` — 5 biến cốt lõi từ SQL Bước 3 & Kiểm định Bước 4):** `Age`, `AnnualIncome`, `LastPurchaseDaysAgo`, `DiscountsAvailed`, `LoyaltyProgram`. Loại bỏ hoàn toàn biến khảo sát cuối phiên `CustomerSatisfaction` (thời điểm T2) để chống rò rỉ dữ liệu tương lai (*Look-ahead Bias*).
> - **Lý do không cần chuẩn hóa `StandardScaler`:** Cả `DecisionTreeClassifier` và `RandomForestClassifier` đều phân nhánh dựa trên ngưỡng thứ tự giá trị (Gini Impurity), bất biến với phép co giãn tuyến tính của đơn vị đo.
> - **Chia tập dữ liệu chuẩn mực:** Sử dụng `stratify=df['PurchaseStatus']` trong `train_test_split` (80% Train = 345.804 dòng / 20% Test = 86.452 dòng) để bảo toàn tỷ lệ chuyển đổi chuẩn `43.34%` trên cả 2 tập.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

# 1. Nạp dữ liệu sạch (432.256 bản ghi) & chọn 5 biến cốt lõi từ Tệp 3-4-5 (Loại bỏ CustomerSatisfaction để chống Data Leakage)
df = pd.read_csv('../Data/customerData_clean.csv' if pd.io.common.file_exists('../Data/customerData_clean.csv') else 'Data/customerData_clean.csv')
features = ['Age', 'AnnualIncome', 'LastPurchaseDaysAgo', 'DiscountsAvailed', 'LoyaltyProgram']
X_train, X_test, y_train, y_test = train_test_split(
    df[features], df['PurchaseStatus'], test_size=0.2, random_state=42, stratify=df['PurchaseStatus']
)

In [2]:
# 2. Huấn luyện 2 mô hình & xuất Bảng So Sánh Hiệu Năng
dt = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=50, max_depth=3, random_state=42).fit(X_train, y_train)

models_df = pd.DataFrame({
    'Mô hình': ['Decision Tree', 'Random Forest'],
    'Accuracy': [round(accuracy_score(y_test, dt.predict(X_test)), 3), round(accuracy_score(y_test, rf.predict(X_test)), 3)],
    'F1-Score': [round(f1_score(y_test, dt.predict(X_test)), 3), round(f1_score(y_test, rf.predict(X_test)), 3)]
})
display(models_df)

,Mô hình,Accuracy,F1-Score
0,Decision Tree,0.874,0.855
1,Random Forest,0.881,0.874


In [3]:
# 3. Xuất Bảng Độ Quan Trọng Của Biến (Feature Importance) & Luật Cây Quyết Định IF-THEN
fi_df = pd.DataFrame({
    'Biến': features,
    'Độ quan trọng': [round(x, 3) for x in rf.feature_importances_]
}).sort_values('Độ quan trọng', ascending=False)
display(fi_df)

# (Tùy chọn khi vấn đáp) In bộ luật phân nhánh IF-THEN từ Cây Quyết Định:
# print(export_text(dt, feature_names=features))

,Biến,Độ quan trọng
2,LastPurchaseDaysAgo,0.946
4,LoyaltyProgram,0.026
1,AnnualIncome,0.020
0,Age,0.006
3,DiscountsAvailed,0.003


In [4]:
# 4. Phân bổ 3 Kịch bản can thiệp kinh doanh dựa trên xác suất P
prob = rf.predict_proba(X_test)[:, 1]
n = len(prob)
c1, c2, c3 = (prob >= 0.70).sum(), ((prob >= 0.30) & (prob < 0.70)).sum(), (prob < 0.30).sum()

print(f"Kịch bản 1 (P >= 70%): {c1:,} khách ({c1/n*100:.1f}%) -> Giữ nguyên giá niêm yết (Không cấp voucher)")
print(f"Kịch bản 2 (30% <= P < 70%): {c2:,} khách ({c2/n*100:.1f}%) -> Tặng voucher kích cầu kịp thời (Tối đa 5 mã)")
print(f"Kịch bản 3 (P < 30%): {c3:,} khách ({c3/n*100:.1f}%) -> Ngắt quảng cáo trả phí (Paid Ads)")

Kịch bản 1 (P >= 70%): 32,268 khách (37.3%) -> Giữ nguyên giá niêm yết (Không cấp voucher)
Kịch bản 2 (30% <= P < 70%): 14,504 khách (16.8%) -> Tặng voucher kích cầu kịp thời (Tối đa 5 mã)
Kịch bản 3 (P < 30%): 39,680 khách (45.9%) -> Ngắt quảng cáo trả phí (Paid Ads)


---
## Đúc Kết Nghiệp Vụ (Đồng Bộ 100% Với Bước 1 – Bước 5)
1. **Mô hình tối ưu:** Random Forest (`max_depth=3`, 50 cây) đạt hiệu năng vượt trội và ổn định hơn so với Decision Tree đơn lẻ (`Accuracy = 0.881` so với `0.874`; `F1-Score = 0.874` so với `0.855`), vượt xa mốc Baseline đoán lớp đa số (`56.66%`).
2. **Đặc trưng chi phối (`Feature Importance`):**
   - `LastPurchaseDaysAgo` chiếm **94.6%** (`0.946`) độ quan trọng — phản ánh chính xác hiện tượng *"Vách đá Recency"* ở Truy vấn SQL 7 (khách mua trong 15 ngày đạt CVR `93.07%`, nhưng quá 60 ngày rơi tự do về `0.07%`).
   - `LoyaltyProgram` đứng thứ 2 với **2.6%** (`0.026`) và `AnnualIncome` đứng thứ 3 với **2.0%** (`0.020`), kết hợp cùng `Age` (**0.6%**) và `DiscountsAvailed` (**0.3%**) làm đòn bẩy phân loại nhóm khách do dự.
3. **Thực thi 3 kịch bản can thiệp thời gian thực (Trên tập Test độc lập `N = 86.452` khách):**
   - **Kịch bản 1 (`P >= 70%` — 32.268 khách, chiếm `37.3%`):** Nhóm khách vừa giao dịch gần đây và có xác suất mua tự nhiên rất cao -> **Giữ nguyên giá niêm yết (Không cấp thêm voucher)** để chống xói mòn biên lợi nhuận.
   - **Kịch bản 2 (`30% <= P < 70%` — 14.504 khách, chiếm `16.8%`):** Nhóm khách đang do dự -> **Tự động kích cầu bằng đặc quyền Loyalty và tặng voucher kịp thời (chặn trần tối đa 5 mã/khách theo SQL 6)**.
   - **Kịch bản 3 (`P < 30%` — 39.680 khách, chiếm `45.9%`):** Nhóm khách ngừng tương tác lâu ngày (`> 60 ngày`) hoặc chỉ xem dạo -> **Ngắt ngay ngân sách quảng cáo trả phí (`Paid Ads` có CVR thấp nhất sàn `43.05%` theo SQL 2)**, chuyển sang kênh tự nhiên chi phí thấp.